In [32]:
import ssl

# Disable SSL verification - because it is annoying
ssl._create_default_https_context = ssl._create_unverified_context

In [2]:
from convokit import Corpus, download

awry_corpus = Corpus(filename=download("conversations-gone-awry-cmv-corpus")) #contains personal attacks
# winning_corpus = Corpus(filename=download("winning-args-corpus")) #conflict resolved
# coarse_discourse_corpus = Corpus(filename=download("reddit-coarse-discourse-corpus")) #use the disagreement label only

Dataset already exists at /Users/priyadcosta/.convokit/downloads/conversations-gone-awry-cmv-corpus


In [3]:
#convert the data into a dataframe
df_awry = awry_corpus.get_utterances_dataframe()
# df_winning = winning_corpus.get_utterances_dataframe()
# df_coarse_discourse = coarse_discourse_corpus.get_utterances_dataframe()

In [30]:
print(df_awry.columns)

Index(['timestamp', 'text', 'speaker', 'reply_to', 'conversation_id',
       'meta.score', 'meta.top_level_comment', 'meta.retrieved_on',
       'meta.gilded', 'meta.gildings', 'meta.subreddit', 'meta.stickied',
       'meta.permalink', 'meta.author_flair_text', 'meta.parsed', 'vectors'],
      dtype='object')


In [4]:
print(df_awry['conversation_id'].nunique())

6842


In [7]:
import topics_over_time as tot

In [8]:
#rename conversation_id to conversation_num
df_awry.rename(columns={'conversation_id': 'conversation_num'}, inplace=True)

#make the conversation numbers numerical (currently they are strings)
tot.convert_convo_nums(df_awry)

print(df_awry)

          timestamp                                               text  \
id                                                                       
cue8y0b  1440446000  (Okay, I've seen this view come up a few times...   
cuec5fs  1440450798  It's not just black and white America though. ...   
cuect48  1440451823  Abstract reasoning is a skill that can be nurt...   
cuedf8c  1440452797  Can we agree that genes account for about 50% ...   
cuedywn  1440453690  Twin studies studies suggest that about 80 per...   
...             ...                                                ...   
e8r0qu2  1540934108  I don't think I need to give reasons why its b...   
e8r0u3b  1540934182  Well you've gotten multiple in this thread the...   
e8r1qwj  1540934931  I just want to say, I'm a little bummed out by...   
e8r1x52  1540935073  I don't see how it's any different in tone the...   
e8r2si1  1540935804  To address your points:\n\nI don't think I'm c...   

                 speaker reply_to  co

In [9]:
#create a small section of the data
df_tiny = df_awry[df_awry['conversation_num'] < 5]

In [18]:
# Import the configuration file (if using the config.py method)
from conflict_features.config import OPENAI_API_KEY

In [34]:
#use the GPT API to tag each sentence as conflict or not

import openai

openai.api_key = OPENAI_API_KEY

def detect_conflict(input_text,question,engine):
    # Define a prompt that instructs the model to provide information about conflict
    prompt = f"{question}\n{input_text}\n\nResponse:"

    # Generate text based on the prompt
    response = openai.Completion.create(
        engine=engine,  # You can choose an appropriate engine
        prompt=prompt,
        max_tokens=100,  # Adjust the max tokens as needed
        temperature=0.7,  # Adjust the temperature for creativity
        top_p=1.0,
        frequency_penalty=0.0,
        presence_penalty=0.0
    )

    # Get the generated text from the model's response
    generated_text = response.choices[0].text

    # Analyze the generated text for indications of conflict (you can define your own criteria)
    if 'text contains conflict' in generated_text.strip():
        return 1 #Conflict detected
    else:
        return 0 #No conflict detected



In [37]:
question = "Which of these indicate conflict amongst the speakers?. Just give the response as 'text contains conflict' or 'text does not contain conflict'."
engine = "text-davinci-003"
df_tiny['conflict_result'] = df_tiny['text'].apply(lambda x: detect_conflict(x,question,engine))

In [38]:
df_tiny.to_csv('awry_data_gpt_labeling.csv')